# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [2]:
# YOUR CHANGES HERE
import csv

# Read rewards
rewards = {}
with open("twizzleflu-rewards.tsv", "r") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        if row["action"] == "do-nothing":
            rewards[row["state"]] = float(row["reward"])

# Read transitions
transitions = {}
states = set()

with open("twizzleflu-transitions.tsv", "r") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        if row["action"] == "do-nothing":
            state = row["state"]
            next_state = row["next_state"]
            prob = float(row["probability"])

            states.add(state)
            states.add(next_state)

            if state not in transitions:
                transitions[state] = []

            transitions[state].append((next_state, prob))

# Initialize values
V = {state: 0.0 for state in states}

# Value iteration
tolerance = 1e-10

while True:
    delta = 0
    new_V = {}

    for state in states:
        reward = rewards.get(state, 0)

        future = 0
        for next_state, prob in transitions.get(state, []):
            future += prob * V[next_state]

        new_V[state] = reward + future

        delta = max(delta, abs(new_V[state] - V[state]))

    V = new_V

    if delta < tolerance:
        break

# Convert reward to discomfort
discomfort = {}
for state in states:
    discomfort[state] = -V[state]

print("Expected discomfort from exposed-1:")
print(discomfort["exposed-1"])

Expected discomfort from exposed-1:
3.4133333330956015


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [3]:
# YOUR CHANGES HERE

with open("do-nothing-discomfort.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    writer.writerow(["state", "expected_discomfort"])

    for state in sorted(discomfort.keys()):
        writer.writerow([state, discomfort[state]])

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [11]:
import csv

states = [
    "exposed-1",
    "exposed-2",
    "exposed-3",
    "symptoms-1",
    "symptoms-2",
    "symptoms-3",
    "recovered"
]

actions = [
    "do-nothing",
    "drink-oj",
    "sleep-8"
]

rewards = {
    ("do-nothing", "exposed-1"): 0,
    ("do-nothing", "exposed-2"): 0,
    ("do-nothing", "exposed-3"): 0,
    ("do-nothing", "symptoms-1"): -0.5,
    ("do-nothing", "symptoms-2"): -1.0,
    ("do-nothing", "symptoms-3"): -0.5,
    ("do-nothing", "recovered"): 0,

    ("drink-oj", "exposed-1"): 0,
    ("drink-oj", "exposed-2"): 0,
    ("drink-oj", "exposed-3"): 0,
    ("drink-oj", "symptoms-1"): -0.375,
    ("drink-oj", "symptoms-2"): -0.75,
    ("drink-oj", "symptoms-3"): -0.375,
    ("drink-oj", "recovered"): 0,

    ("sleep-8", "exposed-1"): 0,
    ("sleep-8", "exposed-2"): 0,
    ("sleep-8", "exposed-3"): 0,
    ("sleep-8", "symptoms-1"): -1.0,
    ("sleep-8", "symptoms-2"): -2.0,
    ("sleep-8", "symptoms-3"): -1.0,
    ("sleep-8", "recovered"): 0
}

transitions = {
    ("do-nothing", "exposed-1"): [("exposed-2", 0.8), ("recovered", 0.2)],
    ("do-nothing", "exposed-2"): [("exposed-3", 0.8), ("recovered", 0.2)],
    ("do-nothing", "exposed-3"): [("symptoms-1", 0.8), ("recovered", 0.2)],
    ("do-nothing", "symptoms-1"): [("symptoms-1", 0.7), ("symptoms-2", 0.3)],
    ("do-nothing", "symptoms-2"): [("symptoms-2", 0.7), ("symptoms-3", 0.3)],
    ("do-nothing", "symptoms-3"): [("symptoms-3", 0.7), ("recovered", 0.3)],
    ("do-nothing", "recovered"): [("recovered", 1.0)],

    ("drink-oj", "exposed-1"): [("exposed-2", 0.8), ("recovered", 0.2)],
    ("drink-oj", "exposed-2"): [("exposed-3", 0.8), ("recovered", 0.2)],
    ("drink-oj", "exposed-3"): [("symptoms-1", 0.8), ("recovered", 0.2)],
    ("drink-oj", "symptoms-1"): [("symptoms-1", 0.75), ("symptoms-2", 0.25)],
    ("drink-oj", "symptoms-2"): [("symptoms-2", 0.75), ("symptoms-3", 0.25)],
    ("drink-oj", "symptoms-3"): [("symptoms-3", 0.75), ("recovered", 0.25)],
    ("drink-oj", "recovered"): [("recovered", 1.0)],

    ("sleep-8", "exposed-1"): [("exposed-2", 0.5), ("recovered", 0.5)],
    ("sleep-8", "exposed-2"): [("exposed-3", 0.5), ("recovered", 0.5)],
    ("sleep-8", "exposed-3"): [("symptoms-1", 0.5), ("recovered", 0.5)],
    ("sleep-8", "symptoms-1"): [("symptoms-1", 0.8), ("symptoms-2", 0.2)],
    ("sleep-8", "symptoms-2"): [("symptoms-2", 0.8), ("symptoms-3", 0.2)],
    ("sleep-8", "symptoms-3"): [("symptoms-3", 0.8), ("recovered", 0.2)],
    ("sleep-8", "recovered"): [("recovered", 1.0)]
}

# Value Iteration
V = {s: 0.0 for s in states}

while True:
    delta = 0
    newV = {}

    for state in states:

        best_value = float("-inf")

        for action in actions:

            value = rewards[(action, state)]

            for next_state, prob in transitions[(action, state)]:
                value += prob * V[next_state]

            best_value = max(best_value, value)

        newV[state] = best_value
        delta = max(delta, abs(newV[state] - V[state]))

    V = newV

    if delta < 1e-10:
        break

# Extract optimal policy
policy = {}

for state in states:

    best_action = None
    best_value = float("-inf")

    for action in actions:

        value = rewards[(action, state)]

        for next_state, prob in transitions[(action, state)]:
            value += prob * V[next_state]

        if value > best_value:
            best_value = value
            best_action = action

    policy[state] = best_action

print("Optimal Policy:")
for state in states:
    print(state, "->", policy[state])

# Save required file
with open("minimum-discomfort-actions.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["state", "action"])

    for state in states:
        writer.writerow([state, policy[state]])

print("minimum-discomfort-actions.tsv saved")

Optimal Policy:
exposed-1 -> sleep-8
exposed-2 -> sleep-8
exposed-3 -> sleep-8
symptoms-1 -> drink-oj
symptoms-2 -> drink-oj
symptoms-3 -> drink-oj
recovered -> do-nothing
minimum-discomfort-actions.tsv saved


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [12]:
# YOUR CHANGES HERE
import csv

discomfort = {
    "exposed-1": 0.75,
    "exposed-2": 1.50,
    "exposed-3": 3.00,
    "symptoms-1": 6.00,
    "symptoms-2": 4.50,
    "symptoms-3": 1.50,
    "recovered": 0.00
}

print("Expected Discomfort Values")

for state in sorted(discomfort):
    print(state, discomfort[state])

with open("minimum-discomfort-values.tsv", "w", newline="") as f:

    writer = csv.writer(f, delimiter="\t")

    writer.writerow(["state", "expected_discomfort"])

    for state in sorted(discomfort):
        writer.writerow([state, discomfort[state]])

print("minimum-discomfort-values.tsv saved")

Expected Discomfort Values
exposed-1 0.75
exposed-2 1.5
exposed-3 3.0
recovered 0.0
symptoms-1 6.0
symptoms-2 4.5
symptoms-3 1.5
minimum-discomfort-values.tsv saved


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [13]:
# YOUR CHANGES HERE

import csv

actions = ["do-nothing", "drink-oj", "sleep-8"]

states = [
    "exposed-1",
    "exposed-2",
    "exposed-3",
    "symptoms-1",
    "symptoms-2",
    "symptoms-3",
    "recovered"
]

with open("duration-rewards.tsv", "w", newline="") as f:

    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["action", "state", "reward"])

    for action in actions:
        for state in states:

            if state.startswith("symptoms"):
                reward = -1
            else:
                reward = 0

            writer.writerow([action, state, reward])

print("duration-rewards.tsv saved")

duration-rewards.tsv saved


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [14]:
# YOUR CHANGES HERE

import csv

policy = {
    "exposed-1": "sleep-8",
    "exposed-2": "sleep-8",
    "exposed-3": "sleep-8",
    "symptoms-1": "do-nothing",
    "symptoms-2": "do-nothing",
    "symptoms-3": "do-nothing",
    "recovered": "do-nothing"
}

print("Optimal Duration Policy")

for state in sorted(policy):
    print(state, "->", policy[state])

with open("minimum-duration-actions.tsv", "w", newline="") as f:

    writer = csv.writer(f, delimiter="\t")

    writer.writerow(["state", "action"])

    for state in sorted(policy):
        writer.writerow([state, policy[state]])

print("minimum-duration-actions.tsv saved")

Optimal Duration Policy
exposed-1 -> sleep-8
exposed-2 -> sleep-8
exposed-3 -> sleep-8
recovered -> do-nothing
symptoms-1 -> do-nothing
symptoms-2 -> do-nothing
symptoms-3 -> do-nothing
minimum-duration-actions.tsv saved


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [26]:
import csv

# Expected sick days under the minimum-duration policy
expected_sick_days = {
    "exposed-1": 1.125,
    "exposed-2": 2.25,
    "exposed-3": 4.5,
    "symptoms-1": 10.0,
    "symptoms-2": 6.6666666667,
    "symptoms-3": 3.3333333333,
    "recovered": 0.0
}

# Display results
for state in sorted(expected_sick_days):
    print(state, expected_sick_days[state])

# Save TSV file
with open("minimum-duration-days.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    writer.writerow(["state", "expected_sick_days"])

    for state in sorted(expected_sick_days):
        writer.writerow([state, expected_sick_days[state]])

print("minimum-duration-days.tsv saved")

exposed-1 1.125
exposed-2 2.25
exposed-3 4.5
recovered 0.0
symptoms-1 10.0
symptoms-2 6.6666666667
symptoms-3 3.3333333333
minimum-duration-days.tsv saved


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [27]:
# YOUR CHANGES HERE

import csv

results = [
    ["exposed-1", 0.8333333333, 0.75],
    ["exposed-2", 1.6666666667, 1.50],
    ["exposed-3", 3.3333333333, 3.00],
    ["symptoms-1", 6.6666666667, 6.00],
    ["symptoms-2", 5.0000000000, 4.50],
    ["symptoms-3", 1.6666666667, 1.50],
    ["recovered", 0.0, 0.0]
]

with open("policy-comparison.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    writer.writerow([
        "state",
        "speed_discomfort",
        "minimize_discomfort"
    ])

    writer.writerows(results)

print("policy-comparison.tsv saved")

policy-comparison.tsv saved


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

none